# Personal news feed based on OpenAI API

In [28]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
from scaper import fetch_website_contents
import json

news_website_urls = [
    "https://ouragan.cd/",
    "https://ouragan.cd/tag/actualites",
    "https://ouragan.cd/rubrique/politique",
    "https://ouragan.cd/rubrique/culture",
    "https://ouragan.cd/rubrique/economie",
    "https://ouragan.cd/rubrique/nation",
    "https://ouragan.cd/rubrique/sport",
    "https://ouragan.cd/rubrique/afp",
    "https://actualite.cd/",
    "https://www.jeuneafrique.com/",
    "https://www.rfi.fr/fr/",
    "https://www.radiookapi.net/",
    "https://7sur7.cd/",
    "https://www.dw.com/fr/rd-congo/t-19027248",
    "https://fr.africanews.com/pays/republique-democratique-du-congo/",
    "https://www.lemonde.fr/congo-rdc/",
    "https://afrikarabia.com/wordpress/",
]

load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY", None)
gemini_base_url = os.getenv("GEMINI_BASE_URL", None)
model_gemini = "gemini-2.5-flash-lite"

if not api_key or not gemini_base_url:
    print("No API key or Gemini base URL found")
else:
    print("environment is set up correctly!")


system_prompt = """
You are a news aggregator focused on the Democratic Republic of Congo.
You are given a dict json object with list of titles and links. structured as.
{
    "titles": ["title1", "title2", "title3"],
    "links": ["link1", "link2", "link3"]
}

And keywords to filter the news by. 

You should return the most relevant titles and links in Markdown format.
* title 1
* title 2
* title 3

* [link 1](link1)
* [link 2](link2)
* [link 3](link3)
"""


def get_news_user_prompt(keywords):
    question = f"""
    Here are the keywords to filter the news by: {", ".join(keywords)}
    """
    get_news_contents = {"data": []}

    for url in news_website_urls:
        print(f"fetching {url}")
        get_news_contents["data"].append(fetch_website_contents(url))
    json_string = json.dumps(get_news_contents, ensure_ascii=False, indent=2)

    question += f"""
    Here are the titles and links to the news: {json_string}
    """

    return question


gemini = OpenAI(base_url=gemini_base_url, api_key=api_key)

question = get_news_user_prompt(["Ukraine", "kabila", "Felix Tshisekedi"])

stream = gemini.chat.completions.create(
    model=model_gemini,
    messages=[
        {"role": "user", "content": question},
        {"role": "system", "content": system_prompt},
    ],
    stream=True,
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content
    response = response.replace("```", "").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)


environment is set up correctly!
fetching https://ouragan.cd/
fetching https://ouragan.cd/tag/actualites
fetching https://ouragan.cd/rubrique/politique
fetching https://ouragan.cd/rubrique/culture
fetching https://ouragan.cd/rubrique/economie
fetching https://ouragan.cd/rubrique/nation
fetching https://ouragan.cd/rubrique/sport
fetching https://ouragan.cd/rubrique/afp
fetching https://actualite.cd/
fetching https://www.jeuneafrique.com/
fetching https://www.rfi.fr/fr/
fetching https://www.radiookapi.net/
fetching https://7sur7.cd/
fetching https://www.dw.com/fr/rd-congo/t-19027248
fetching https://fr.africanews.com/pays/republique-democratique-du-congo/
fetching https://www.lemonde.fr/congo-rdc/
fetching https://afrikarabia.com/wordpress/


* Ukrainiens, Russes et Américains ont tenu leurs premiers pourparlers à Abou Dhabi
* Jaynet Kabila réclame la libération du coordonnateur de la Fondation Mzee LD Kabila
* Félix Tshisekedi est à Paris, un tête-à-tête avec Emmanuel Macron prévu à 13H00
* RDC : Joseph Kabila condamné à mort, Emmanuel Ramazani Shadary arrêté à Kinshasa
* RDC : le M23 soutenu par le Rwanda pénètre dans la ville stratégique d’Uvira

* [/2026/01/ukrainiens-russes-et-americains-ont-tenu-leurs-premiers-pourparlers-a-abou-dhabi](/2026/01/ukrainiens-russes-et-americains-ont-tenu-leurs-premiers-pourparlers-a-abou-dhabi)
* [/2026/01/jaynet-kabila-reclame-la-liberation-du-coordonnateur-de-la-fondation-mzee-ld-kabila](/2026/01/jaynet-kabila-reclame-la-liberation-du-coordonnateur-de-la-fondation-mzee-ld-kabila)
* [/2026/01/23/felix-tshisekedi-est-paris-un-tete-tete-avec-emmanuel-macron-prevu-13h00](/2026/01/23/felix-tshisekedi-est-paris-un-tete-tete-avec-emmanuel-macron-prevu-13h00)
* [/2025/10/16/rdc-l-ancien-president-joseph-kabila-condamne-a-mort-rassemble-des-opposants-a-nairobi](/2025/10/16/rdc-l-ancien-president-joseph-kabila-condamne-a-mort-rassemble-des-opposants-a-nairobi)
* [/2025/12/13/rdc-le-m23-sempare-duvira-malgre-laccord-de-paix-de-washington](/2025/12/13/rdc-le-m23-sempare-duvira-malgre-laccord-de-paix-de-washington)